In [48]:
# Importing required Python Libraries
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
)

In [49]:
# Loading required variables from environment
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
OPENAI_API_KEY = openai_key

In [50]:
# We are defining the schema type for checks, so LLM will be forced to provide output in this format
# for more details https://pydantic.dev/docs/validation/latest/get-started/
class GDPRCheckOutput(BaseModel):
    is_gdpr_check: bool
    reasoning: str

In [51]:
# You are defining the Guardrail agent with its logic and behaviour
guardrail_agent = Agent(
    name="Guardrail Agent",
    instructions="""
    Analyze the user's location or request context and return a structured response.
    1. If the customer is from the UK or Europe: Set is_gdpr_check to False (allow).
    2. If the customer is from Asia or the US: Set is_gdpr_check to True (block).
    3. Provide clear reasoning for your decision.
    """,
    model="gpt-4o-mini",
    output_type=GDPRCheckOutput
)

In [53]:
# This function decides based on the Guardrail bahvior and output whether to allow the request to proceed or block it, by raising an exception
@input_guardrail
async def GDPRGuardrail(
        ctx: RunContextWrapper[None],
        agent: Agent,
        input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(guardrail_agent, input, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output.reasoning,
        tripwire_triggered=result.final_output.is_gdpr_check,
    )


In [54]:
# Main Agent to talk to Users

agent = Agent(
    name="EV Sales Assistant",
    instructions="You must check all EV benchmarks available to you and advise customer on top 3 models to choose with its 4 year resale value",
    input_guardrails=[GDPRGuardrail],
    model="gpt-4o-mini"
)

In [57]:
# Executing Agent
async def main() -> None:
    try:
        result = await Runner.run(agent, "I am from India, Can you tell me the latest EV models available in UK ?")
        print("Request allowed. Result:", result.final_output)
    except InputGuardrailTripwireTriggered:
        print("Guardrail blocked the request.")


# Run the async function in Jupyter using await
await main()

Guardrail blocked the request.
